**LIBRARY IMPORTS AND PARAMETER DEFINITIONS**

In [24]:
import os
from tqdm.notebook import trange
import numpy as np
import time
import secrets

SCOPETYPE = 'OPENADC'
PLATFORM = 'CW308_STM32F3'
CRYPTO_TARGET = 'TINYAES128C'
SS_VER = 'SS_VER_1_1'

# Firmware implementations
unmasked_asm = "keccak_unmasked_Opt_ASSEMBLY"  
masked_ti_1st_order_asm = "keccak_masked_TIFirstOrder3shares_1stOrder_Opt_ASSEMBLY"   # 1st-order TI, 3 shares
masked_ti_2nd_order_asm = "keccak_masked_TIFirstOrder3shares_2ndOrder_Opt_ASSEMBLY"   # 1st+2nd-order TI, 3 shares

base_dir_hex = ".\\..\\..\\firmware\\mcu"

# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%% DEFINE PARAMETERS HERE %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
base_dir_traces = "C:\\Users\\moreno\\Desktop\\TrazasKeccak"   # Base directory where traces will be stored
experiment_name = "DATASET_KECCAK_TIDOM"                   # Dataset subfolder name

masking_list = [masked_ti_2nd_order_asm]                   # List of firmware implementations to evaluate
remasking_list = ["no-remasking"]

iteration_start = 0                                          # Index of the first trace batch
iteration_end = 20000000000                                  # Index of the last trace batch

iterations_per_full_trace = 11                               # Number of concatenated sub-traces per full trace (3 for unmasked, 11 for masked)
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%


**INITIALIZATION OF CHIPWHISPERER PLATFORM**

In [17]:
# CHIPWHISPERER-LITE CW308 PLATFORM INITIALIZATION
%run "../Setup_Scripts/Setup_Generic.ipynb"

INFO: Found ChipWhisperer😍
scope.clock.adc_freq                     changed from 29538459                  to 26251201                 
scope.clock.adc_rate                     changed from 29538459.0                to 26251201.0               


**CODES COMPILATION**

In [18]:
%%bash -s "$PLATFORM" "$CRYPTO_TARGET" "$SS_VER"

cd ../../firmware/mcu/keccak_unmasked_Opt_ASSEMBLY
make PLATFORM=$1 CRYPTO_TARGET=$2 SS_VER=$3 -j

cd ../keccak_masked_TIFirstOrder3shares_1stOrder_Opt_ASSEMBLY/no-remasking
make PLATFORM=$1 CRYPTO_TARGET=$2 SS_VER=$3 -j

cd ../../keccak_masked_TIFirstOrder3shares_2ndOrder_Opt_ASSEMBLY/no-remasking
make PLATFORM=$1 CRYPTO_TARGET=$2 SS_VER=$3 -j

SS_VER set to SS_VER_1_1
SS_VER set to SS_VER_1_1
arm-none-eabi-gcc (15:9-2019-q4-0ubuntu1) 9.2.1 20191025 (release) [ARM/arm-9-branch revision 277599]
Copyright (C) 2019 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

mkdir -p objdir-CW308_STM32F3 
.
Welcome to another exciting ChipWhisperer target build!!
.
.
.
.
.
.
.
.
.
.
.
Assembling: Keccak1600NoOpt.s
Compiling:
.
Compiling:
arm-none-eabi-gcc -c -mcpu=cortex-m4 -mthumb -I. Keccak1600NoOpt.s -o objdir-CW308_STM32F3/Keccak1600NoOpt.o
Compiling:
Compiling:
Compiling:
Compiling:
Compiling:
Compiling:
Compiling:
Compiling:
-en     KeccakFW.c ...
Compiling:
-en     .././hal//stm32f3/stm32f3_hal.c ...
Assembling: .././hal//stm32f3/stm32f3_startup.S
-en     .././hal//stm32f3/stm32f3xx_hal_msp.c ...
-en     .././hal//stm32f3/stm32f3xx_it.c ...
-en     .././hal//stm32f3/stm32f3xx_hal_gpio.c ...
-en     ../

KeccakFW.c:280:6: warning: conflicting types for 'create_state'
  280 | void create_state(uint8_t* plaintext, int len)
      |      ^~~~~~~~~~~~
KeccakFW.c:223:5: note: previous implicit declaration of 'create_state' was here
  223 |     create_state(pt, len);
      |     ^~~~~~~~~~~~
KeccakFW.c: In function 'get_mask':
KeccakFW.c:215:5: warning: 'memcpy' forming offset [3, 16] is out of the bounds [0, 2] of object 'initial_mask' with type 'uint8_t[2]' {aka 'unsigned char[2]'} [-Warray-bounds]
  215 |     memcpy(&initial_mask[0], &mask[0], 16);
      |     ^~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
KeccakFW.c:13:9: note: 'initial_mask' declared here
   13 | uint8_t initial_mask[2];
      |         ^~~~~~~~~~~~


-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
.
LINKING:
-en     KeccakFW-CW308_STM32F3.elf ...
-e Done!
.
.
.
.
.
Creating load file for Flash: KeccakFW-CW308_STM32F3.hex
arm-none-eabi-objcopy -O ihex -R .eeprom -R .fuse -R .lock -R .signature KeccakFW-CW308_STM32F3.elf KeccakFW-CW308_STM32F3.hex
Creating load file for EEPROM: KeccakFW-CW308_STM32F3.eep
Creating load file for Flash: KeccakFW-CW308_STM32F3.bin
arm-none-eabi-objcopy -O binary -R .eeprom -R .fuse -R .lock -R .signature KeccakFW-CW308_STM32F3.elf KeccakFW-CW308_STM32F3.bin
Creating Extended Listing: KeccakFW-CW308_STM32F3.lss
arm-none-eabi-objcopy -j .eeprom --set-section-flags=.eeprom="alloc,load" \
--change-section-lma .eeprom=0 --no-change-warnings -O ihex KeccakFW-CW308_STM32F3.elf KeccakFW-CW308_STM32F3.eep || exit 0
arm-none-eabi-objdump -h -S -z KeccakFW-CW308_STM32F3.elf > KeccakFW-CW308_STM32F3.lss
Creating Symbol Table: KeccakFW-CW308_STM32F3.sym
arm-none-eabi-nm -n KeccakFW-CW308_STM32

KeccakFW.c:304:6: warning: conflicting types for 'create_state'
  304 | void create_state(uint8_t* plaintext, int len)
      |      ^~~~~~~~~~~~
KeccakFW.c:230:5: note: previous implicit declaration of 'create_state' was here
  230 |     create_state(pt, len);
      |     ^~~~~~~~~~~~
KeccakFW.c:326:6: warning: conflicting types for 'mask_state'
  326 | void mask_state(void)
      |      ^~~~~~~~~~
KeccakFW.c:231:5: note: previous implicit declaration of 'mask_state' was here
  231 |     mask_state();
      |     ^~~~~~~~~~
KeccakFW.c:344:6: warning: conflicting types for 'unmask_state'
  344 | void unmask_state(void)
      |      ^~~~~~~~~~~~
KeccakFW.c:270:5: note: previous implicit declaration of 'unmask_state' was here
  270 |     unmask_state();
      |     ^~~~~~~~~~~~
KeccakFW.c: In function 'get_mask':
KeccakFW.c:222:5: warning: 'memcpy' forming offset [3, 16] is out of the bounds [0, 2] of object 'initial_mask' with type 'uint8_t[2]' {aka 'unsigned char[2]'} [-Warray-bounds]
 

-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
.
LINKING:
-en     KeccakFW-CW308_STM32F3.elf ...
-e Done!
.
.
.
.
.
Creating load file for Flash: KeccakFW-CW308_STM32F3.bin
arm-none-eabi-objcopy -O binary -R .eeprom -R .fuse -R .lock -R .signature KeccakFW-CW308_STM32F3.elf KeccakFW-CW308_STM32F3.bin
Creating load file for EEPROM: KeccakFW-CW308_STM32F3.eep
arm-none-eabi-objcopy -j .eeprom --set-section-flags=.eeprom="alloc,load" \
--change-section-lma .eeprom=0 --no-change-warnings -O ihex KeccakFW-CW308_STM32F3.elf KeccakFW-CW308_STM32F3.eep || exit 0
Creating load file for Flash: KeccakFW-CW308_STM32F3.hex
Creating Extended Listing: KeccakFW-CW308_STM32F3.lss
arm-none-eabi-objcopy -O ihex -R .eeprom -R .fuse -R .lock -R .signature KeccakFW-CW308_STM32F3.elf KeccakFW-CW308_STM32F3.hex
arm-none-eabi-objdump -h -S -z KeccakFW-CW308_STM32F3.elf > KeccakFW-CW308_STM32F3.lss
Creating Symbol Table: KeccakFW-CW308_STM32F3.sym
arm-none-eabi-nm -n KeccakFW-CW308_STM32

KeccakFW.c:304:6: warning: conflicting types for 'create_state'
  304 | void create_state(uint8_t* plaintext, int len)
      |      ^~~~~~~~~~~~
KeccakFW.c:230:5: note: previous implicit declaration of 'create_state' was here
  230 |     create_state(pt, len);
      |     ^~~~~~~~~~~~
KeccakFW.c:326:6: warning: conflicting types for 'mask_state'
  326 | void mask_state(void)
      |      ^~~~~~~~~~
KeccakFW.c:231:5: note: previous implicit declaration of 'mask_state' was here
  231 |     mask_state();
      |     ^~~~~~~~~~
KeccakFW.c:344:6: warning: conflicting types for 'unmask_state'
  344 | void unmask_state(void)
      |      ^~~~~~~~~~~~
KeccakFW.c:270:5: note: previous implicit declaration of 'unmask_state' was here
  270 |     unmask_state();
      |     ^~~~~~~~~~~~
KeccakFW.c: In function 'get_mask':
KeccakFW.c:222:5: warning: 'memcpy' forming offset [3, 16] is out of the bounds [0, 2] of object 'initial_mask' with type 'uint8_t[2]' {aka 'unsigned char[2]'} [-Warray-bounds]
 

-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
-e Done!
.
LINKING:
-en     KeccakFW-CW308_STM32F3.elf ...
-e Done!
.
.
.
.
.
Creating load file for Flash: KeccakFW-CW308_STM32F3.hex
Creating load file for Flash: KeccakFW-CW308_STM32F3.bin
arm-none-eabi-objcopy -O ihex -R .eeprom -R .fuse -R .lock -R .signature KeccakFW-CW308_STM32F3.elf KeccakFW-CW308_STM32F3.hex
arm-none-eabi-objcopy -O binary -R .eeprom -R .fuse -R .lock -R .signature KeccakFW-CW308_STM32F3.elf KeccakFW-CW308_STM32F3.bin
Creating Extended Listing: KeccakFW-CW308_STM32F3.lss
Creating load file for EEPROM: KeccakFW-CW308_STM32F3.eep
arm-none-eabi-objdump -h -S -z KeccakFW-CW308_STM32F3.elf > KeccakFW-CW308_STM32F3.lss
arm-none-eabi-objcopy -j .eeprom --set-section-flags=.eeprom="alloc,load" \
--change-section-lma .eeprom=0 --no-change-warnings -O ihex KeccakFW-CW308_STM32F3.elf KeccakFW-CW308_STM32F3.eep || exit 0
Creating Symbol Table: KeccakFW-CW308_STM32F3.sym
arm-none-eabi-nm -n KeccakFW-CW308_STM32

**DEFINITION OF FUNCTIONS FOR TRACES ACQUISITIONS**

In [19]:
import os
import h5py
import numpy as np
from tqdm import trange
import secrets

# ================== CONFIGURATION ==================
TRACES_PER_FILE = 5000
SAMPLES_PER_ITERATION = 24400

# ================== HDF5 HANDLING ==================
def open_hdf5_file(file_index, experiment_name, base_dir_traces,
                   total_samples_per_trace):

    file_path = os.path.join(
        base_dir_traces,
        experiment_name,
        f"{experiment_name}_traces_part{file_index:03d}.h5"
    )

    f = h5py.File(file_path, 'w')  # Always create a new file

    dset_traces = f.create_dataset(
        'traces',
        shape=(TRACES_PER_FILE, total_samples_per_trace),
        maxshape=(TRACES_PER_FILE, total_samples_per_trace),
        dtype='float32'
    )

    dset_plaintexts = f.create_dataset(
        'plaintexts',
        shape=(TRACES_PER_FILE, 16),
        dtype='uint8'
    )

    dset_masks = f.create_dataset(
        'masks',
        shape=(TRACES_PER_FILE, 16),
        dtype='uint8'
    )

    dset_seeds = f.create_dataset(
        'seeds',
        shape=(TRACES_PER_FILE, 32),
        dtype='uint8'
    )

    return f, dset_traces, dset_plaintexts, dset_masks, dset_seeds


def capture_traces(use_seed, experiment_name):
    scope.gain.db = 0.1

    scope.adc.samples = 24400
    scope.adc.timeout = 2

    ktp = cw.ktp.Basic()
    key, text = ktp.next()

    fixed_plaintext = bytes([108, 81, 56, 195, 111, 10, 64, 204,
                             0, 43, 197, 36, 103, 202, 212, 146])

    target.baud = 38400

    os.makedirs(os.path.join(base_dir_traces, experiment_name), exist_ok=True)

    total_samples = iterations_per_full_trace * SAMPLES_PER_ITERATION

    # ---- File state ----
    file_index = 0
    trace_index = 0

    f, dset_traces, dset_plaintexts, dset_masks, dset_seeds = open_hdf5_file(
        file_index, experiment_name, base_dir_traces, total_samples
    )

    try:
        for _ in range(iteration_start, iteration_end):
            full_trace = np.empty(total_samples, dtype=np.float32)

            for i in trange(10, desc='Capturing traces'):
                scope.arm()
                target.flush()

                # -------- PLAINTEXT SELECTION --------
                # Fixed vs. Random
                if i % 2:
                    key, text = ktp.next()
                else:
                    text = fixed_plaintext

                if use_seed:
                    # -------- CHACHA20 SEED GENERATION --------
                    external_seed = [int.from_bytes(secrets.token_bytes(1), 'little') for _ in range(32)]

                    for k in range(2):
                        target.simpleserial_write('s', external_seed[k * 16:(k + 1) * 16])
                        _ = target.simpleserial_read('r', 16)

                    # -------- PLAINTEXT INITIAL MASKING --------
                    mask = [int.from_bytes(secrets.token_bytes(1), 'little') for _ in range(16)]
                    masked_text = [m ^ t for m, t in zip(mask, text)]

                    # -------- TRANSMIT MASKED PLAINTEXT --------
                    target.simpleserial_write('k', mask)
                    _ = target.simpleserial_read('r', 16)

                    for k in range(iterations_per_full_trace):
                        scope.adc.offset = k * 24400
                        trace = cw.capture_trace(scope, target, masked_text)
                        start = k * SAMPLES_PER_ITERATION
                        end = start + SAMPLES_PER_ITERATION
                        full_trace[start:end] = trace.wave
                else:
                    # -------- UNMASKED PLAINTEXT TRANSMISSION --------
                    for k in range(iterations_per_full_trace):
                        scope.adc.offset = k * 24400
                        trace = cw.capture_trace(scope, target, text)
                        start = k * SAMPLES_PER_ITERATION
                        end = start + SAMPLES_PER_ITERATION
                        full_trace[start:end] = trace.wave

                # -------- DATA STORAGE --------
                dset_traces[trace_index, :] = full_trace
                dset_plaintexts[trace_index, :] = np.frombuffer(text, dtype=np.uint8)

                if use_seed:
                    dset_masks[trace_index, :] = mask
                    dset_seeds[trace_index, :] = external_seed

                trace_index += 1

                # # ---------READING FINAL STATE---------
                print(f"-----TRAZA {i}-----")
                print("Plaintext usado: ", text)
                for j in range(25):
                    target.simpleserial_write('r',[j, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
                    msg_res = target.simpleserial_read('m', 8)
                    print("Lane ", j,": ",' '.join(f'{b:02X}' for b in msg_res))
                print("\n\n")

                # -------- FILE ROTATION --------
                if trace_index >= TRACES_PER_FILE:
                    f.close()
                    file_index += 1
                    trace_index = 0
                    f, dset_traces, dset_plaintexts, dset_masks, dset_seeds = open_hdf5_file(
                        file_index, experiment_name, base_dir_traces, total_samples
                    )
    finally:
        f.close()


**MICROCONTROLLER FLASHING AND INITIALIZATION OF TRACES ACQUISITION**

In [26]:
# FLASH STM32F3 TARGET AND RUN TRACE ACQUISITION
for masking in masking_list:
    masking_path = os.path.normpath(os.path.join(base_dir_hex, masking))

    if masking in ["keccak_unmasked_NoOpt_ASSEMBLY", "keccak_unmasked_Opt_ASSEMBLY"]:
        for filename in os.listdir(masking_path):
            if filename.endswith(".hex"):
                hex_path = os.path.normpath(os.path.join(masking_path, filename))
                cw.program_target(scope, prog, hex_path.format(PLATFORM))
                capture_traces(use_seed=False, experiment_name="exp_" + masking)
    else:
        for remasking in remasking_list:
            remasking_path = os.path.normpath(os.path.join(base_dir_hex, masking, remasking))
            for filename in os.listdir(remasking_path):
                if filename.endswith(".hex"):
                    hex_path = os.path.normpath(os.path.join(remasking_path, filename))
                    cw.program_target(scope, prog, hex_path.format(PLATFORM))
                    capture_traces(
                        use_seed=True,
                        experiment_name="exp_" + masking + "_" + remasking
                    )


Detected known STMF32: STM32F302xB(C)/303xB(C)
Extended erase (0x44), this can take ten seconds or more
Attempting to program 38343 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 38343 bytes


Capturing traces:   0%|          | 0/10 [00:00<?, ?it/s]

-----TRAZA 0-----
Plaintext usado:  b'lQ8\xc3o\n@\xcc\x00+\xc5$g\xca\xd4\x92'
Lane  0 :  01 CA 95 88 66 F0 99 73
Lane  1 :  35 72 26 B4 F5 31 C1 8E
Lane  2 :  FE 4F 25 7D EC 1C EF 88
Lane  3 :  02 4A FF AD 1F A0 54 8B
Lane  4 :  13 C3 E3 11 DA BC 2C 90
Lane  5 :  6C 31 26 32 CD BB 61 31
Lane  6 :  95 C4 0D 56 06 98 42 4A
Lane  7 :  81 24 35 47 28 25 84 A5
Lane  8 :  0E EC 9A 9C BE AC 86 E7
Lane  9 :  F5 B2 9A 6D ED 57 8C C6
Lane  10 :  07 D9 30 53 C0 A2 37 C5
Lane  11 :  2D B3 74 E6 2A 5B FA 72
Lane  12 :  82 1A 79 1B 01 35 13 D5
Lane  13 :  99 92 49 3C 69 C0 BA 1D
Lane  14 :  78 E0 1F BF 07 15 7C 9F
Lane  15 :  06 E3 06 04 1F 27 6E DF
Lane  16 :  45 64 EF 88 6E 95 07 36
Lane  17 :  49 13 16 5D DC 4A 97 C4
Lane  18 :  0F 95 2B 18 90 DE B2 B5
Lane  19 :  38 1F 0E F1 C3 FD A0 C8
Lane  20 :  2F 6C 8C C2 B2 58 50 FA
Lane  21 :  2C FA 32 AC D0 95 52 6B
Lane  22 :  43 13 2A 7C F1 5C ED 87
Lane  23 :  E9 BD 76 5A E5 3F B1 A0
Lane  24 :  D8 8A F5 DE 53 90 DD AE



-----TRAZA 1-----
Plaintext u

KeyboardInterrupt: 